In [ ]:
import pyecap
import os
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import scipy.signal
from plotly.subplots import make_subplots
import scipy.io as sio
from scipy.signal import find_peaks
from scipy.stats import zscore
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from flexNIRs.fnirs_functions import *

In [ ]:
meta_index = 8
metaDF = pd.read_excel(r'D:\Data\TCD\20260220_TCD_QC\meta.xlsx')
tank_folder = r'D:\Data\TCD\20260220_TCD_QC\CVP_SingleCh_Ramp-260220\\'

raw_ecg = pyecap.Ephys(tank_folder + metaDF.loc[meta_index, 'Tank'], stores = 'ECGG')
raw_ecg = raw_ecg.remove_ch(['ECGG 2','ECGG 3', 'ECGG 4'])

raw_tcd = pyecap.Ephys(tank_folder + metaDF.loc[meta_index, 'Tank'], stores = 'TCDD')
raw_tcd_fir = raw_tcd.filter_fir(cutoff = (1,10), width = 1, pass_zero = 'bandpass')

stim = pyecap.Stim(tank_folder + metaDF.loc[meta_index, 'Tank'])
stimDF = stim.parameters
#stimDF

In [ ]:
"""Generate arrays and Down-sample for plotting"""
DS_factor = 20

ecg_d = raw_ecg.array[0,:].compute()
ecg_DS = signal.decimate(ecg_d, DS_factor, zero_phase = True)
ecg_f = filter_hr(raw_ecg, cutoff = 15, downsample=False, check_plot=False)[0] *1e-6

ecg_DS_f = signal.decimate(ecg_f, DS_factor, zero_phase = True)
scaler = MinMaxScaler(feature_range = (0,1))
ecg_DS_norm = np.squeeze(scaler.fit_transform(ecg_DS_f.reshape(-1,1)))

tcd_d = signal.medfilt(raw_tcd.array[0,:].compute(), kernel_size = 3)  #Applies 3 window median filter to remove TCD data drops
tcd_fir = raw_tcd_fir.array[0,:].compute()
tcd_f = filter_hr(raw_tcd, cutoff = 100, downsample=False, check_plot=False)[0] *1e-6

tcd_DS = signal.decimate(tcd_d, DS_factor, zero_phase = True)
tcd_fir_DS = signal.decimate(tcd_fir, DS_factor, zero_phase = True)
tcd_DS_f = filter_hr(raw_tcd, cutoff = 10, downsample=True, downsample_factor = DS_factor, check_plot=False)[0] *1e-6

time = raw_ecg.time()
time_ds = np.arange(len(tcd_DS)) * (DS_factor / raw_tcd.sample_rate)
ds_dt = DS_factor / raw_tcd.sample_rate
idx_ds = np.arange(len(time_ds))

In [ ]:
"""Perform Calculations on DS arrays"""
dvt_f = np.gradient(tcd_DS_f, DS_factor / raw_tcd.sample_rate) ** 3
dvt_r = np.gradient(tcd_DS, DS_factor / raw_tcd.sample_rate) ** 3

In [ ]:
"""Plots"""
fig = make_subplots(specs = [[{'secondary_y' : True}]])
fig.update_layout(spikedistance = 0)

fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS_f, name='Gauss Filtered DS'))
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_fir_DS, name='FIR Filtered DS'))
#fig.add_trace(go.Scattergl(x = time, y = tcd_d, name='Raw Signal'))
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS, name='Raw TCD Signal DS'))
fig.add_trace(go.Scattergl(x = time_ds, y = dvt_f, name='1st Dvt - Filtered Data'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = time_ds, y = dvt_r, name='1st Dvt - Raw Data'), secondary_y = True)

#fig.add_trace(go.Scattergl(x = time_ds, y = ecg_DS, name='Raw ECG DS'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = time_ds, y = ecg_DS_f, name='Filtered ECG DS'), secondary_y = True)
fig.add_trace(go.Scattergl(x = time_ds, y = ecg_DS_norm, name='Filtered ECG DS'))

fig.show()

In [ ]:
TCD_peaks,_ = find_peaks(dvt_f, height = 40, distance = 100)
ECG_peaks,_ = find_peaks(ecg_DS_norm, height = 0.8, distance = 100)

In [ ]:
"""Plots"""
fig = make_subplots(specs = [[{'secondary_y' : True}]])
fig.update_layout(spikedistance = 0)

fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS_f, name='Gauss Filtered DS'))
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_fir_DS, name='FIR Filtered DS'))
#fig.add_trace(go.Scattergl(x = time, y = tcd_d, name='Raw Signal'))
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS, name='Raw TCD Signal DS'))
fig.add_trace(go.Scattergl(x = time_ds, y = dvt_f, name='1st Dvt - Filtered Data'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = time_ds, y = dvt_r, name='1st Dvt - Raw Data'), secondary_y = True)

#fig.add_trace(go.Scattergl(x = time_ds, y = ecg_DS, name='Raw ECG DS'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = time_ds, y = ecg_DS_f, name='Filtered ECG DS'), secondary_y = True)
fig.add_trace(go.Scattergl(x = time_ds, y = ecg_DS_norm, name='Filtered ECG DS'))
fig.add_trace(go.Scattergl(x = time_ds[TCD_peaks], y = dvt_f[TCD_peaks], mode = 'markers'), secondary_y = True)
fig.add_trace(go.Scattergl(x = time_ds[ECG_peaks], y = ecg_DS_norm[ECG_peaks], mode = 'markers'))

fig.show()

In [ ]:
"""Find time delay between peaks in ECG data vs. TCD data

Assuming the ECG data is very clean:
- Find closest TCD peak(s) to each ECG peak -- and which occurred after the ECG peak
- Find mean/variance and exclude outliers to remove doubles?

"""
TCD_peaks,_ = find_peaks(dvt_f, height = 40, distance = 100)
ECG_peaks,_ = find_peaks(ecg_DS_norm, height = 0.8, distance = 100)

idxLIST = []
tcd_peaks_2 = []
for tcd_peak in TCD_peaks:

    if tcd_peak < ECG_peaks[0]:
        pass
    else:
        #Find closest (preceding) ECG peak
        diff = np.abs(ECG_peaks - tcd_peak)
        idx = np.argmin(diff) #Index of the closest ECG peak from the ECG_peaks array

        if ECG_peaks[idx] > tcd_peak:
            print(idx, 'shift')
            idx = idx - 1

        tcd_peaks_2.append(tcd_peak)
        idxLIST.append(idx)

ds = pd.DataFrame(np.column_stack((ECG_peaks[idxLIST], tcd_peaks_2)), columns = ['ECG peak', 'TCD peak'])
ds['delta Sample'] = ds['TCD peak'] - ds['ECG peak']

"""Calculate HR from ECG peaks"""

"""Find Outliers based on Interquartile Method"""
Q1 = ds['delta Sample'].quantile(0.25)
Q3 = ds['delta Sample'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
ds['Outlier IQR'] = (ds['delta Sample'] < lower_bound) | (ds['delta Sample'] > upper_bound)

"""Find Outliers based on Z-score"""
ds['Z Scores'] = np.abs(zscore(ds['delta Sample']))
ds['Outlier Z Scores'] = ds['Z Scores'] > 1.5
ds

In [ ]:
"""Plots"""
dbls = ds[ds['ECG peak'].duplicated(keep=False)].index

fig = make_subplots(specs = [[{'secondary_y' : True}]])
fig.update_layout(spikedistance = 0)

# TCD signal
#fig.add_trace(go.Scattergl(x = idx_ds, y = tcd_DS_f, name='Gauss Filtered DS'))

#TCD First Derivative and Markers
fig.add_trace(go.Scattergl(x = idx_ds, y = dvt_f, name='1st Dvt - Filtered Data'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = idx_ds[TCD_peaks], y = dvt_f[TCD_peaks], mode = 'markers'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = time_ds, y = dvt_r, name='1st Dvt - Raw Data'), secondary_y = True)

# ECG Signal and Markers
fig.add_trace(go.Scattergl(x = idx_ds, y = ecg_DS_norm, name='Filtered ECG DS'))
fig.add_trace(go.Scattergl(x = idx_ds[ECG_peaks], y = ecg_DS_norm[ECG_peaks], mode = 'markers'))

#Duplicates
fig.add_trace(go.Scattergl(x = idx_ds[ds.loc[dbls, 'TCD peak']], y = dvt_f[ds.loc[dbls, 'TCD peak']],mode = 'markers', name='Double TCD Counts'), secondary_y = True)

fig.show()

In [ ]:
dbls = ds[ds['ECG peak'].duplicated(keep=False)]
drop_idx = dbls.loc[dbls['Outlier IQR'] == True]
ds.drop(drop_idx.index, inplace=True)
ds

In [ ]:
ds['ECG TCD Delay'] = ds['delta Sample'].mul(ds_dt)
ds

In [ ]:
"""Plots"""
fig = make_subplots(specs = [[{'secondary_y' : True}]])
fig.update_layout(spikedistance = 0)

# TCD signal
#fig.add_trace(go.Scattergl(x = time_ds, y = tcd_DS_f, name='Filtered TCD DS'))

#TCD First Derivative and Markers
#fig.add_trace(go.Scattergl(x = idx_ds, y = dvt_f, name='1st Dvt - Filtered Data'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = idx_ds[TCD_peaks], y = dvt_f[TCD_peaks], mode = 'markers'), secondary_y = True)
#fig.add_trace(go.Scattergl(x = time_ds, y = dvt_r, name='1st Dvt - Raw Data'), secondary_y = True)

# ECG Signal and Markers
#fig.add_trace(go.Scattergl(x = time_ds, y = ecg_DS_norm, name='Filtered ECG DS'))
#fig.add_trace(go.Scattergl(x = idx_ds[ECG_peaks], y = ecg_DS_norm[ECG_peaks], mode = 'markers'))

#Delay
fig.add_trace(go.Scattergl(x = time_ds[ds['TCD peak']], y = ds['ECG TCD Delay'], name='ECG-TCD Delay'))#, secondary_y = True)

for param in stimDF.index:
    fig.add_vrect(x0 = stimDF.loc[param, 'onset time (s)'], x1 = stimDF.loc[param, 'offset time (s)'],fillcolor = 'gray', opacity = 0.25)

fig.add_trace(go.Scattergl(x = ecgDF['Time (s)'], y = ecgDF['b2b bpm']), secondary_y = True)

fig.show()

In [ ]:
ecg_f2 = filter_hr(raw_ecg, cutoff = 15, downsample=False, check_plot=False)
ecgDF = calc_hr(ecg_f2, peak_height = 59, fs = raw_ecg.sample_rate, check_plot=True)
ecgDF

In [ ]:
ecgDF